# 工具调用

- 工具实际上是明确了输入输出的可调用函数，因此，工具调用(Tool Calling)也叫函数调用(Function Calling)

### 两种调用方式

In [3]:
# 方式1 直接调用 适合测试

from langchain_core.tools import tool

@tool
def get_weater(city:str)->str:
    """
    获取指定城市的天气信息

    参数：
        city：城市名称
    返回：
        天气信息字符串
    """
    return city+"晴天15度"

result=get_weater.invoke({"city":"Nanjing"})
print(result)

Nanjing晴天15度


In [6]:
# 方式2 绑定到模型 让ai调用

from langchain.chat_models import  init_chat_model
from langchain_core.tools import  tool
from dotenv import load_dotenv
import os

load_dotenv(override=True)
model=init_chat_model(
    model="deepseek-flash",
    model_name="deepseek"
)

@tool
def get_weater(city:str)->str:
    """
    获取指定城市的天气信息

    参数：
        city：城市名称
    返回：
        天气信息字符串
    """
    return city+"晴天15度"

## 绑定工具
model_with_tools=model.bind_tools([get_weater])

#AI决定是否调用工具

response1=model_with_tools.invoke("南京天气如何")
#response2=model_with_tools.invoke("1+1=?")

#检查AI是否要调用工具

if response1.tool_calls:
    print("AI要调用工具:",response1.tool_calls)
else:
    print(response1.content)

AI要调用工具: [{'name': 'get_weater', 'args': {'city': '南京'}, 'id': 'call_00_t9pxN1S0W9HawT9WugS44836', 'type': 'tool_call'}]


## 通过message流转看工具调用


In [ ]:
from langchain.chat_models import  init_chat_model
from langchain_core.tools import  tool
from dotenv import load_dotenv
import os

load_dotenv(override=True)
model=init_chat_model(
    model="deepseek-flash",
    model_name="deepseek"
)

In [7]:
# 不使用@tool修饰
from langchain.messages import HumanMessage, ToolMessage

def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗"
# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])
messages = [
    HumanMessage("今天北京天气如何")
]
# 模型生成调用工具请求
response = model_with_tools.invoke(messages)
# 添加AIMessage
messages.append(response)
tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
# 拼接出ToolMessage实例
        tool_response = ToolMessage(
        content=get_weather(**tool_call["args"]),
        tool_call_id=tool_call["id"],
        name=tool_call["name"]
        )
messages.append(tool_response)

print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_ET_R8yc0BPjnZqlYuCAxs3P2839)
 Call ID: call_00_ET_R8yc0BPjnZqlYuCAxs3P2839
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗
=====================> messages <=====================
final_response: 
content='今天北京天气**晴朗** ☀️\n\n建议出门注意防晒，可以适当补充水分。如果需要了解气温、风力或未来几天的预报，可以告诉我～' additional_kwargs={'refusal': None, 'reasoning_content': 'The tool returned "北京天气晴朗" (Beijing weather is sunny/clear). I should relay this to the user.'} response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 344, 'total_tokens': 403, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 24, 'rejected_prediction

In [8]:
# 使用@tool修饰

from langchain.messages import HumanMessage, ToolMessage
@tool
def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗~"
# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])
messages = [
    HumanMessage("今天北京天气如何")
]
# 模型生成调用工具请求
response = model_with_tools.invoke(messages)
# 添加AIMessage
messages.append(response)
tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 返回的是ToolMessage类型消息
        tool_response = get_weather.invoke(tool_call)
print(type(tool_response))
messages.append(tool_response)
print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_H9xTSt5aAT5phYnvj8Qj1110)
 Call ID: call_00_H9xTSt5aAT5phYnvj8Qj1110
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗~
=====================> messages <=====================
final_response: 
content='今天北京天气晴朗 ☀️，适合外出活动。建议注意防晒，如果长时间在户外可以带上太阳镜和遮阳帽。需要我帮你查其他城市或未来几天的天气吗？' additional_kwargs={'refusal': None, 'reasoning_content': ''} response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 360, 'total_tokens': 403, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'a

![image.png](images/model-tool.png)

## 工具调用流程

1. 模型绑定工具
2. 模型生成调用工具请求
3. 根据模型调用返回的工具名手动执行工具
4. 将工具结果返回给模型生成最终结果

## 工具定义方式1：不使用@tool

### 模型绑定工具并发送请求

In [10]:
    from langchain.chat_models import init_chat_model
from dotenv import  load_dotenv
from rich import print as rprint
import os

load_dotenv(override=True)

model=init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek"
)

def get_weater(city:str)->str:
    return f"{city}天气晴朗"

model_with_tools=model.bind_tools([get_weather])
response=model_with_tools.invoke("今天南京天气")

rprint(response)

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The user asks about today\'s weather in Nanjing. I should call get_weather with city 
"南京".'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 60,
            'prompt_tokens': 289,
            'total_tokens': 349,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 21,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128},
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 161
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
        'id': '989b55e0-700d-4ede-bd89-36e8f29b969b',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a1c057-f8e5-7033-b9f8-522e56ad6a29-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '南京'},
            'id': 'call_00_3iOekSGF8GIwI4q886B58804',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 289,
        'output_tokens': 60,
        'total_tokens': 349,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 21}
    }
)

## 2.convert_to_openai_tool
执行model.bind_tools([get_weather])，底层最终会调用convert_to_openai_tool生成工具描述。所 以我们可以直接调用后者查看解析后的工具描述。

In [11]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city: str):
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

结果字段说明：
(1) type：定义当前数据节点必须是什么数据类型。常见类型有 string, number, integer, boolean, object, array, null。object即是json对象。
(2) properties：用于定义JSON 对象（Object）中可以包含哪些属性（键），以及每个属性对应的值类型和说明。
(3) required：当 type为 "object"时使用，是一个数组，列出了对象中必须存在的属性名

### 2.1 description说明
convert_to_openai_tool会从docstring(文档字符串)加载工具的描述信息，上面的案例中，docstring为空，所以抽取的description为空。

In [14]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

### 2.2参数说明
使用 Args:、Returns:、Raises:等关键字，这种方式可读性强。Agent通过工具的这些注释来理解工具的用途和调用时机，因此清晰、准确的文档字符串是工具能被正确调用的前提。

In [16]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city:天气名称
    """

    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {'city': {'description': '天气名称', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

### 2.3参数默认值

如果参数没有默认值，则会包含在required对应的列表中。
反之，则参数的描述信息会包含default字段，并且不会出现在required列表中。

In [17]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city: str="北京"):
    """
    天气查询工具
    Args:
        city: 城市名称
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具\nArgs:\n    city: 城市名称',
        'parameters': {'properties': {'city': {'default': '北京', 'type': 'string'}}, 'type': 'object'}
    }
}

In [18]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(dt: str, city: str="北京"):
    """
    天气查询工具
    Args:
        dt: 日期
        city: 城市名称
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具\nArgs:\n    dt: 日期\n    city: 城市名称',
        'parameters': {
            'properties': {'dt': {'type': 'string'}, 'city': {'default': '北京', 'type': 'string'}},
            'required': ['dt'],
            'type': 'object'
        }
    }
}

## 工具定义方式2：使用@tool

使用@tool装饰器修饰，可以自动将普通 Python 函数转化为智能体可调用的工具

##  3.1 自定义工具描述description

- 情况1：仅提供docstring信息

在bind_tools()调用时，先将函数封装为BaseTool类型的对象，再传递给convert_to_openai_tool 函数，生成工具的描述

In [20]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from langchain.tools import tool
@tool
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

- 情况2：添加工具描述：description

@tool的参数description可以更改工具描述，优先级高于docstring的函数说明

In [21]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint
@tool(description="根据城市名称查询当日天气的工具")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '根据城市名称查询当日天气的工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

- 情况三：解析docstring：parse_docstring

当我们没有向@tool传递description参数时，默认情况下，tool会将docstring整体视为description，如下

In [22]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool
def get_weather(city: str, units: str = "celsius", include_forecast: bool =
False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报
    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来五日的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报\nArgs:\n    city: 城市\n    units: 
气温单位，可选：celsius-摄氏度，fahrenheit-华氏度\n    include_forecast: 是否包含未来五日的天气预报',
        'parameters': {
            'properties': {
                'city': {'type': 'string'},
                'units': {'default': 'celsius', 'type': 'string'},
                'include_forecast': {'default': False, 'type': 'boolean'}
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 更改工具名称：name_or_callable

默认情况，使用函数名作为工具名称，但可以向@tool 传参name_or_callable，以更改工具名称

In [25]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from langchain.tools import tool
@tool(name_or_callable="getWeather")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

说明：@tool中参数name_or_callable名称可以省略

In [26]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from langchain.tools import tool
@tool("getWeather")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

##  自定义args_schema